# Lab 5: Transfer Learning - Cats vs Dogs Classification

## 🎯 Learning Objectives

By the end of this lab, you will:
- Understand transfer learning and why it works
- Use a pre-trained CNN (Convolutional Neural Network) as feature extractor
- Add a custom classification head for binary classification
- Work with image datasets from Kaggle
- Implement data augmentation for better generalization
- Evaluate model performance with appropriate metrics

**What You'll Build:** A cats vs dogs classifier achieving >90% accuracy using transfer learning

**Reference:** [Kaggle Cats vs Dogs CNN](https://www.kaggle.com/code/sachinpatil1280/cats-vs-dogs-image-classification-using-cnn-95/notebook)

**Note:** PyTorch and torchvision are pre-installed in Google Colab!

**Setup:** Check GPU availability (Colab provides free T4 GPU!)

In [ ]:
import torch

# Detect device (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✓ GPU detected! Training will be fast.")
else:
    print("\n⚠️  No GPU detected. Training will be slower.")
    print("In Colab: Runtime → Change runtime type → Hardware accelerator → T4 GPU")

## Part 1: What is Transfer Learning?

### The Big Idea

Instead of training a CNN from scratch (which requires millions of images and days of training), we:
1. **Use a pre-trained model** (trained on ImageNet - 14M images, 1000 classes)
2. **Freeze the feature extractor** (keep learned patterns: edges, textures, shapes)
3. **Add a new classification head** (train only this part for our specific task)

### Why Transfer Learning Works

**Low-level features** (edges, corners, colors) are universal:
- A ResNet trained on ImageNet learned to detect edges, textures, shapes
- These same features are useful for cats vs dogs!
- No need to relearn them

**Benefits:**
- ✅ **Faster training** (minutes vs days)
- ✅ **Less data needed** (thousands vs millions)
- ✅ **Better accuracy** (pre-trained features are high quality)
- ✅ **Less compute** (fine-tune on CPU or small GPU)

### Architecture

```
Input Image (3×224×224)
        ↓
Pre-trained ResNet18 (frozen) ← Trained on ImageNet
        ↓
Feature Vector (512 dimensions)
        ↓
Custom Head (trainable) ← We train this!
  - FC Layer 1: 512 → 128
  - ReLU
  - Dropout
  - FC Layer 2: 128 → 1
  - Sigmoid
        ↓
Output (0 = cat, 1 = dog)
```

## Part 2: Dataset Setup

### Kaggle Cats vs Dogs Dataset

**Dataset:** [Dogs vs Cats on Kaggle](https://www.kaggle.com/c/dogs-vs-cats/data)
- 25,000 labeled images
- 12,500 cats, 12,500 dogs
- Various sizes and orientations

### Downloading the Dataset

**Option 1: Direct Kaggle API (Recommended)**

In [ ]:
# Install kaggle package
!pip install -q kaggle

# Upload your kaggle.json (API token) to Colab
# Get it from: https://www.kaggle.com/settings → API → Create New API Token
# Then upload to Colab files panel

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✓ Kaggle setup complete!")
print("Note: If kaggle.json not found, upload it from Kaggle settings.")

In [ ]:
# Download dataset
!kaggle competitions download -c dogs-vs-cats

# Unzip
!unzip -q dogs-vs-cats.zip
!unzip -q train.zip

print("✓ Dataset downloaded and extracted!")

**Option 2: Manual Upload**

1. Download dataset from Kaggle manually
2. Upload train.zip to Colab
3. Run: `!unzip -q train.zip`

### Organize Dataset Structure

The downloaded files are named like: `cat.123.jpg`, `dog.456.jpg`

We'll organize them into folders:
```
data/
  train/
    cats/
      cat.0.jpg
      cat.1.jpg
    dogs/
      dog.0.jpg
      dog.1.jpg
  val/
    cats/
    dogs/
```

In [ ]:
import os
import shutil
from pathlib import Path

# Create directory structure
os.makedirs('data/train/cats', exist_ok=True)
os.makedirs('data/train/dogs', exist_ok=True)
os.makedirs('data/val/cats', exist_ok=True)
os.makedirs('data/val/dogs', exist_ok=True)

# Get all files from train directory
train_dir = Path('train')
all_files = list(train_dir.glob('*.jpg'))

print(f"Total images found: {len(all_files)}")

# Separate cats and dogs
cat_files = [f for f in all_files if 'cat' in f.name]
dog_files = [f for f in all_files if 'dog' in f.name]

print(f"Cats: {len(cat_files)}, Dogs: {len(dog_files)}")

# Split into train/val (80/20)
# Hint: Use first 80% for training, last 20% for validation
cat_train = cat_files[:int(0.8*len(cat_files))]
cat_val = cat_files[int(0.8*len(cat_files)):]
dog_train = dog_files[:int(0.8*len(dog_files))]
dog_val = dog_files[int(0.8*len(dog_files)):]

print(f"\nTrain: {len(cat_train)} cats, {len(dog_train)} dogs")
print(f"Val: {len(cat_val)} cats, {len(dog_val)} dogs")

# Copy files to organized structure
for f in cat_train:
    shutil.copy(f, 'data/train/cats/')
for f in cat_val:
    shutil.copy(f, 'data/val/cats/')
for f in dog_train:
    shutil.copy(f, 'data/train/dogs/')
for f in dog_val:
    shutil.copy(f, 'data/val/dogs/')

print("\n✓ Dataset organized into train/val folders!")

### Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

# Sample some cat and dog images
cat_samples = list(Path('data/train/cats').glob('*.jpg'))[:5]
dog_samples = list(Path('data/train/dogs').glob('*.jpg'))[:5]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Show cats
for i, img_path in enumerate(cat_samples):
    img = Image.open(img_path)
    axes[0, i].imshow(img)
    axes[0, i].set_title('Cat', color='blue', fontweight='bold')
    axes[0, i].axis('off')

# Show dogs
for i, img_path in enumerate(dog_samples):
    img = Image.open(img_path)
    axes[1, i].imshow(img)
    axes[1, i].set_title('Dog', color='red', fontweight='bold')
    axes[1, i].axis('off')

plt.suptitle('Sample Images from Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 3: Data Loading and Preprocessing

### Image Preprocessing Requirements

Pre-trained models expect specific input:
1. **Size:** 224×224 pixels
2. **Normalization:** Mean=[0.485, 0.456, 0.406], Std=[0.229, 0.224, 0.225] (ImageNet stats)
3. **Format:** Tensor with shape (batch, 3, 224, 224)

### Data Augmentation

To improve generalization, we randomly transform training images:
- Random horizontal flip
- Random rotation (±10 degrees)
- Random crop and resize
- Color jitter (brightness, contrast)

**API Reference:**
- [torchvision.transforms](https://pytorch.org/vision/stable/transforms.html)
- [ImageFolder dataset](https://pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html)

In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data transformations
# Hint: Training needs augmentation, validation doesn't

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])  # ImageNet stats
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder('data/train', transform=train_transform)
val_dataset = datasets.ImageFolder('data/val', transform=val_transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")
print(f"Class to index: {train_dataset.class_to_idx}")

### Create Data Loaders

Data loaders handle batching and shuffling:

**Batch Size Considerations:**
- Too large: Out of memory
- Too small: Slow training, noisy gradients
- Typical: 32-64 for images

**API Reference:** [torch.utils.data.DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

In [ ]:
# Create data loaders
# Hint: Use batch_size=32, shuffle=True for training, shuffle=False for validation

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Test loading a batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}  (batch_size, channels, height, width)")
print(f"Labels shape: {labels.shape}")
print(f"Sample labels: {labels[:5].tolist()}  (0=cat, 1=dog)")

### Visualize Augmented Images

In [ ]:
# Visualize what augmentation does
import numpy as np

def denormalize(tensor):
    """Reverse ImageNet normalization for visualization."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

# Get a batch
images, labels = next(iter(train_loader))

# Show first 4 images
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)  # Clip to valid range
    axes[i].imshow(img)
    label_name = 'Cat' if labels[i] == 0 else 'Dog'
    axes[i].set_title(label_name, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('Augmented Training Images', fontweight='bold')
plt.show()

## Exercise 1: Build Transfer Learning Model

### Your Task

Build a model using transfer learning:
1. Load a pre-trained ResNet18
2. Freeze all parameters (we don't want to retrain the feature extractor)
3. Replace the final layer with a custom classification head

### Architecture Hints

**Pre-trained Model:**
- Use `torchvision.models.resnet18(pretrained=True)`
- Original has 1000-class output (ImageNet)
- We need binary output (cat vs dog)

**Classification Head:**
```
Input: 512 features (from ResNet18)
  ↓
Linear(512 → 128)
  ↓
ReLU
  ↓
Dropout(0.5)  ← Prevents overfitting
  ↓
Linear(128 → 1)
  ↓
Sigmoid  ← Output between 0 and 1
```

**Loss Function:** BCEWithLogitsLoss (combines sigmoid + binary cross entropy)

**API Reference:**
- [torchvision.models.resnet18](https://pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
- [nn.Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html)
- [BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)

### Starter Code

In [ ]:
import torch.nn as nn
import torchvision.models as models

class CatDogClassifier(nn.Module):
    """Transfer learning model for binary classification."""
    
    def __init__(self):
        super().__init__()
        
        # TODO: Load pre-trained ResNet18
        # self.backbone = models.resnet18(pretrained=True)
        # Note: In newer PyTorch, use: models.resnet18(weights='DEFAULT')
        
        # TODO: Freeze backbone parameters
        # Hint: for param in self.backbone.parameters():
        #           param.requires_grad = False
        
        # TODO: Replace final fully connected layer
        # ResNet18 has self.backbone.fc with 512 inputs
        # Replace it with your custom head:
        # self.backbone.fc = nn.Sequential(
        #     nn.Linear(512, 128),
        #     nn.ReLU(),
        #     nn.Dropout(0.5),
        #     nn.Linear(128, 1)  # Binary output
        # )
        
        raise NotImplementedError("Implement model architecture")
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input tensor (batch, 3, 224, 224)
        
        Returns:
            Output tensor (batch, 1) - logits before sigmoid
        
        TODO:
        Simply pass through backbone:
        return self.backbone(x)
        """
        raise NotImplementedError("Implement forward pass")

### Test Your Model

In [ ]:
# Create model
model = CatDogClassifier()

# Check architecture
print("Model Architecture:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}  (only classification head)")
print(f"Frozen parameters: {total_params - trainable_params:,}  (ResNet18 backbone)")

# Test forward pass
test_input = torch.randn(2, 3, 224, 224)  # Batch of 2 images
test_output = model(test_input)
print(f"\nInput shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}  (should be [2, 1])")
assert test_output.shape == (2, 1), f"Expected shape (2, 1), got {test_output.shape}"
print("\n✓ Model works!")

## Exercise 2: Implement Training Loop

### Training Setup

**Loss Function:** `BCEWithLogitsLoss`
- Combines sigmoid + binary cross-entropy
- More numerically stable than separate sigmoid + BCELoss
- Expects raw logits (no sigmoid in model)

**Optimizer:** Adam
- Learning rate: 0.001 (typical for transfer learning)
- Only optimize trainable parameters

**Training Loop Pattern:**
```python
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        # Forward
        outputs = model(images)
        loss = criterion(outputs.squeeze(), labels.float())
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

### Your Task

Implement the complete training loop with:
- Loss tracking
- Accuracy calculation
- Progress printing

### Starter Code

In [ ]:
import torch.optim as optim

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = CatDogClassifier().to(device)

# TODO: Define loss function
# criterion = nn.BCEWithLogitsLoss()

# TODO: Define optimizer (only trainable parameters!)
# optimizer = optim.Adam(model.parameters(), lr=0.001)
# Note: Since we froze backbone, only head parameters will be optimized

# Training settings
num_epochs = 10

# Track metrics
train_losses = []
train_accs = []
val_accs = []

# TODO: Implement training loop
#
# for epoch in range(num_epochs):
#     model.train()
#     running_loss = 0.0
#     correct = 0
#     total = 0
#     
#     for images, labels in train_loader:
#         images = images.to(device)
#         labels = labels.to(device)
#         
#         # Forward pass
#         outputs = model(images).squeeze()  # Remove dimension
#         loss = criterion(outputs, labels.float())
#         
#         # Backward pass
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         
#         # Track metrics
#         running_loss += loss.item()
#         predictions = (torch.sigmoid(outputs) > 0.5).long()
#         correct += (predictions == labels).sum().item()
#         total += labels.size(0)
#     
#     # Calculate epoch metrics
#     epoch_loss = running_loss / len(train_loader)
#     epoch_acc = correct / total
#     train_losses.append(epoch_loss)
#     train_accs.append(epoch_acc)
#     
#     print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Acc: {epoch_acc*100:.2f}%")

print("\nUncomment the code above to train!")

### Expected Training Progress

With transfer learning, you should see:
```
Epoch [1/10], Loss: 0.3521, Acc: 85.23%
Epoch [2/10], Loss: 0.2134, Acc: 91.45%
Epoch [3/10], Loss: 0.1523, Acc: 94.12%
...
Epoch [10/10], Loss: 0.0823, Acc: 97.34%
```

**Note:** Training is fast because we only train the classification head!

## Exercise 3: Implement Validation

### Evaluation Metrics for Binary Classification

**Accuracy:** Overall correctness
- `accuracy = (TP + TN) / (TP + TN + FP + FN)`

**Precision:** Of predicted positives, how many are actually positive?
- `precision = TP / (TP + FP)`

**Recall:** Of actual positives, how many did we find?
- `recall = TP / (TP + FN)`

**F1 Score:** Harmonic mean of precision and recall
- `F1 = 2 * (precision * recall) / (precision + recall)`

Where:
- TP = True Positives (predicted dog, actually dog)
- TN = True Negatives (predicted cat, actually cat)
- FP = False Positives (predicted dog, actually cat)
- FN = False Negatives (predicted cat, actually dog)

### Your Task

Implement validation with multiple metrics:

**API Reference:** [sklearn.metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# TODO: Implement validation
#
# model.eval()
# all_preds = []
# all_labels = []
#
# with torch.no_grad():
#     for images, labels in val_loader:
#         images = images.to(device)
#         labels = labels.to(device)
#         
#         # Forward pass
#         outputs = model(images).squeeze()
#         
#         # Get predictions
#         predictions = (torch.sigmoid(outputs) > 0.5).long()
#         
#         # Collect all predictions and labels
#         all_preds.extend(predictions.cpu().numpy())
#         all_labels.extend(labels.cpu().numpy())
#
# # Calculate metrics
# accuracy = accuracy_score(all_labels, all_preds)
# precision = precision_score(all_labels, all_preds)
# recall = recall_score(all_labels, all_preds)
# f1 = f1_score(all_labels, all_preds)
#
# print(f"\nValidation Results:")
# print(f"Accuracy:  {accuracy*100:.2f}%")
# print(f"Precision: {precision*100:.2f}%")
# print(f"Recall:    {recall*100:.2f}%")
# print(f"F1 Score:  {f1*100:.2f}%")

print("\nUncomment the code above to validate!")

### Target Performance

A well-trained model should achieve:
- **Accuracy:** > 90%
- **Precision:** > 88%
- **Recall:** > 88%
- **F1 Score:** > 89%

If metrics are lower:
- Train for more epochs
- Try different learning rate (0.0001 or 0.01)
- Add more data augmentation
- Try larger classification head

## Part 4: Visualizing Results

### Training Curves

In [ ]:
# Uncomment after training:

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# # Plot loss
# ax1.plot(train_losses)
# ax1.set_xlabel('Epoch')
# ax1.set_ylabel('Loss')
# ax1.set_title('Training Loss')
# ax1.grid(True, alpha=0.3)

# # Plot accuracy
# ax2.plot([acc*100 for acc in train_accs], label='Train')
# if val_accs:
#     ax2.plot([acc*100 for acc in val_accs], label='Validation')
# ax2.set_xlabel('Epoch')
# ax2.set_ylabel('Accuracy (%)')
# ax2.set_title('Accuracy')
# ax2.legend()
# ax2.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

### Confusion Matrix

In [ ]:
# Uncomment after validation:

# import seaborn as sns

# cm = confusion_matrix(all_labels, all_preds)

# plt.figure(figsize=(6, 5))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
#             xticklabels=['Cat', 'Dog'], 
#             yticklabels=['Cat', 'Dog'])
# plt.xlabel('Predicted')
# plt.ylabel('True')
# plt.title('Confusion Matrix')
# plt.show()

# print(f"\nTrue Negatives (Cat→Cat): {cm[0,0]}")
# print(f"False Positives (Cat→Dog): {cm[0,1]}")
# print(f"False Negatives (Dog→Cat): {cm[1,0]}")
# print(f"True Positives (Dog→Dog): {cm[1,1]}")

### Visualize Predictions

In [ ]:
# Uncomment after training:

# # Get a batch from validation set
# model.eval()
# images, labels = next(iter(val_loader))
# images = images.to(device)

# with torch.no_grad():
#     outputs = model(images).squeeze()
#     predictions = (torch.sigmoid(outputs) > 0.5).long()

# # Show first 8 predictions
# fig, axes = plt.subplots(2, 4, figsize=(12, 6))
# for i, ax in enumerate(axes.flat):
#     img = denormalize(images[i].cpu()).permute(1, 2, 0).numpy()
#     img = np.clip(img, 0, 1)
#     ax.imshow(img)
#     
#     true_label = 'Cat' if labels[i] == 0 else 'Dog'
#     pred_label = 'Cat' if predictions[i].cpu() == 0 else 'Dog'
#     correct = predictions[i].cpu() == labels[i]
#     
#     color = 'green' if correct else 'red'
#     ax.set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontweight='bold')
#     ax.axis('off')

# plt.suptitle('Predictions (Green=Correct, Red=Wrong)', fontsize=14, fontweight='bold')
# plt.tight_layout()
# plt.show()

## Exercise 4: Analyze Model Performance

### Understanding Predictions

Let's look at:
1. **Confidence scores** (how certain is the model?)
2. **Misclassifications** (which images are hardest?)
3. **Per-class performance** (better at cats or dogs?)

### Your Task

Implement confidence analysis:

In [ ]:
# TODO: Get predictions with confidence scores
#
# model.eval()
# all_probs = []  # Confidence scores
# all_preds = []
# all_labels = []
#
# with torch.no_grad():
#     for images, labels in val_loader:
#         images = images.to(device)
#         outputs = model(images).squeeze()
#         probs = torch.sigmoid(outputs)
#         predictions = (probs > 0.5).long()
#         
#         all_probs.extend(probs.cpu().numpy())
#         all_preds.extend(predictions.cpu().numpy())
#         all_labels.extend(labels.cpu().numpy())
#
# all_probs = np.array(all_probs)
# all_preds = np.array(all_preds)
# all_labels = np.array(all_labels)

print("\nUncomment to analyze predictions!")

### Confidence Distribution

In [ ]:
# Uncomment after getting predictions:

# plt.figure(figsize=(10, 4))

# plt.subplot(1, 2, 1)
# plt.hist(all_probs[all_labels==0], bins=30, alpha=0.7, label='Cats', color='blue')
# plt.hist(all_probs[all_labels==1], bins=30, alpha=0.7, label='Dogs', color='red')
# plt.xlabel('Predicted Probability (Dog)')
# plt.ylabel('Count')
# plt.title('Confidence Distribution')
# plt.legend()
# plt.axvline(0.5, color='black', linestyle='--', label='Decision Boundary')

# # Show correct vs incorrect
# plt.subplot(1, 2, 2)
# correct_mask = (all_preds == all_labels)
# plt.hist(all_probs[correct_mask], bins=30, alpha=0.7, label='Correct', color='green')
# plt.hist(all_probs[~correct_mask], bins=30, alpha=0.7, label='Incorrect', color='red')
# plt.xlabel('Predicted Probability (Dog)')
# plt.ylabel('Count')
# plt.title('Correct vs Incorrect Predictions')
# plt.legend()

# plt.tight_layout()
# plt.show()

# print("\nInterpretation:")
# print("- Peaks near 0 and 1 = confident predictions")
# print("- Values near 0.5 = uncertain predictions")
# print("- Incorrect predictions often have lower confidence")

### Find Hardest Examples

In [ ]:
# Uncomment after getting predictions:

# # Find misclassified images
# incorrect_mask = (all_preds != all_labels)
# incorrect_indices = np.where(incorrect_mask)[0]

# if len(incorrect_indices) > 0:
#     print(f"Found {len(incorrect_indices)} misclassifications")
#     print(f"Error rate: {len(incorrect_indices)/len(all_labels)*100:.2f}%")
#     
#     # Show 4 hardest mistakes
#     # Hardest = most confident but wrong
#     confidence = np.abs(all_probs - 0.5)  # Distance from decision boundary
#     incorrect_confidence = confidence[incorrect_mask]
#     hardest_indices = incorrect_indices[np.argsort(incorrect_confidence)[-4:]]
#     
#     # TODO: Visualize these hardest mistakes
#     # Hint: Load images from val_dataset using hardest_indices
#     # Show true label vs predicted label with confidence score
# else:
#     print("Perfect validation! No mistakes.")

## Bonus: Fine-Tuning

### What is Fine-Tuning?

After training the classification head, we can **unfreeze** some backbone layers and train them with a **very small learning rate**.

**Why?**
- Adapt pre-trained features to our specific task
- Can improve accuracy by 1-3%

**Risks:**
- Can overfit if not careful
- Takes longer to train
- Need to use smaller learning rate (0.0001)

### Try Fine-Tuning (Optional)

In [ ]:
# Uncomment to try fine-tuning:

# # Unfreeze last few layers of ResNet
# for param in model.backbone.layer4.parameters():
#     param.requires_grad = True

# # Create new optimizer with small learning rate
# optimizer_ft = optim.Adam([
#     {'params': model.backbone.layer4.parameters(), 'lr': 0.0001},  # Backbone: very small LR
#     {'params': model.backbone.fc.parameters(), 'lr': 0.001}         # Head: normal LR
# ])

# # Train for a few more epochs
# # (Copy training loop from Exercise 2, but use optimizer_ft)

print("Fine-tuning is optional - your model should already work well!")

## Summary

### What You've Learned

✅ **Transfer learning:** Leverage pre-trained models for new tasks

✅ **Feature extraction:** Use frozen backbone as feature extractor

✅ **Custom heads:** Add task-specific layers on top

✅ **Image preprocessing:** Resize, normalize, augment

✅ **Data loaders:** Efficient batching and loading

✅ **Binary classification:** BCEWithLogitsLoss for binary tasks

✅ **Evaluation metrics:** Accuracy, precision, recall, F1 score

✅ **Real dataset:** Kaggle Dogs vs Cats (25k images)

### Journey Complete: From Scratch to Production

**Lab 1:** Built arrays from pure Python → Understood NumPy internals

**Lab 2:** Built computation graphs + autograd → Understood automatic differentiation

**Lab 3:** Built neural networks from scratch → Understood MLP architecture

**Lab 4:** Used PyTorch on MNIST → Understood production frameworks

**Lab 5:** Transfer learning on real images → Understood modern deep learning

### Key Insights

**Why transfer learning is powerful:**
- Pre-trained models learned universal visual features
- 10-100x faster than training from scratch
- Works with limited data (thousands vs millions)
- State-of-the-art results with minimal compute

**When to use it:**
- ✅ Limited training data (<100k samples)
- ✅ Limited compute budget
- ✅ Similar task to pre-training (e.g., ImageNet → cats/dogs)
- ✅ Need fast prototyping

**When to train from scratch:**
- Very different domain (medical images, satellite imagery)
- Massive dataset available (millions of samples)
- Unlimited compute budget

### Next Steps

You're now equipped to:
- Use any pre-trained model (ResNet, VGG, EfficientNet, Vision Transformers)
- Adapt models to custom tasks
- Work with real-world image datasets
- Deploy models to production

**Try these projects:**
- Multi-class classification (10+ classes)
- Object detection (YOLO, Faster R-CNN)
- Image segmentation (U-Net)
- Vision Transformers (ViT)

**Congratulations!** 🎉 You've mastered deep learning from first principles to production-ready systems!